# Inferência no Colab (GPU) — Prompt-Engineering-for-KE

Corre o `three_shot_RAG` (446 prompts) com o Llama-3-8B-Instruct em 4 bits na GPU do Colab e guarda também as
log-probabilidades dos tokens. Antes de começar: **Runtime → Change runtime type → GPU (T4)**.

Também precisas de: (1) acesso aprovado ao `meta-llama/Meta-Llama-3-8B-Instruct` no Hugging Face,
(2) um token *read* guardado nos **Secrets** do Colab (ícone da chave) com o nome `HF_TOKEN`
e com "Notebook access" ligado.

In [ ]:
!nvidia-smi

In [ ]:
# Repositório e branch (a branch tem de estar publicada no GitHub com git push)
REPO_URL = "https://github.com/Luis-FO/Prompt-Engineering-for-KE.git"
BRANCH = "rui/colab"

import os
os.chdir("/content")
if not os.path.isdir("kl"):
    !git clone -b {BRANCH} {REPO_URL} kl
os.chdir("/content/kl")
!git log --oneline -1

In [ ]:
# Dependências (o torch já vem no Colab). N
!pip install -q -U transformers accelerate bitsandbytes tqdm

In [ ]:
# Token do Hugging Face a partir dos Secrets do Colab
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# Guardar no Google Drive: se o Colab desligar, o script retoma onde ficou ao voltares a correr a célula
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kl_out_llama"

## 1. Teste rápido (3 prompts)

In [ ]:
!python 2_inference_module/run_inference_local_HF.py --limit 3 --batch_size 3 --out_dir /content/out_teste

In [ ]:
import json
d = json.load(open("/content/out_teste/three_shot_RAG_prompts_test.json"))
for x in d:
    print(repr(x["generated_text"][:300]))
    print("tokens:", len(x["token_ids"]), "| logprob média:", round(sum(x["token_logprobs"]) / max(1, len(x["token_logprobs"])), 3))
    print()

## 2. Corrida completa (446 prompts)
Demora (provavelmente 1 a 2 horas na T4; é uma estimativa). Se o Colab desligar, volta a correr as células
de configuração e esta: o script retoma a partir do que já está no Drive.
Se der falta de memória, baixa `--batch_size` para 2 ou 1.

In [ ]:
!python 2_inference_module/run_inference_local_HF.py --batch_size 4 --out_dir {OUT}

## 3. Post-processamento e avaliação (F1 deve ficar perto de 0,10)

In [ ]:
!python 2_inference_module/post_process_generated_string.py --input_folder {OUT} --out_dir {OUT}_pp
!python 3_evaluation_module/evaluation_reference_to_annotations/evaluate.py --input_dir {OUT}_pp --gold_file 1_data_module/1_data_preprocessing/RED-fm/test.jsonl --out_dir {OUT}_eval

## 4. Descarregar os resultados (opcional, já estão no Drive)

In [ ]:
!cd /content/drive/MyDrive && zip -qr kl_out_llama.zip kl_out_llama kl_out_llama_pp kl_out_llama_eval
from google.colab import files
files.download("/content/drive/MyDrive/kl_out_llama.zip")